# PS1: margin rules, reputation information, and household opportunity
This self-contained Colab notebook compares fixed and intermediary-chosen margin rules, checks coordination-cost participation thresholds, and exposes one planned reputation demand shifter. All outputs are synthetic. No behavioral study has been run.

The proposed experiment varies margin rule × seller-history information while holding listed price, product and quality evidence, quantity, and delivery fixed. Change `rho_R` only after estimating the seller-history treatment effect.

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Params:
    w: float = 6.0
    c: float = 4.0
    K: float = 12.0
    effort_cost: float = 30.0
    demand_scale: float = 12.0
    choke_margin: float = 4.0
    rho_R: float = 0.0  # assumed demand lift for verified history; not yet estimated

def quantity(m, e, p, verified=False):
    lift = p.rho_R if verified else 0.0
    return p.demand_scale * e * max(0.0, p.choke_margin-m) * (1+lift)

def intermediary_profit(m, e, p, verified=False):
    return m*quantity(m,e,p,verified)-p.effort_cost*e*e

def optimize_fixed(p, verified=False):
    candidates=((intermediary_profit(1.0,i/100,p,verified),1.0,i/100) for i in range(101))
    best=max(candidates,key=lambda x:x[0])
    return best[1],best[2]

def optimize_chosen(p, verified=False):
    candidates=((intermediary_profit(j*.05,i/100,p,verified),j*.05,i/100) for j in range(81) for i in range(101))
    best=max(candidates,key=lambda x:x[0])
    return best[1],best[2]

def evaluate(rule, info, p):
    verified=(info=='Verified')
    m,e=optimize_fixed(p,verified) if rule=='Fixed' else optimize_chosen(p,verified)
    qjoin=quantity(m,e,p,verified)
    farmer_if_join=(p.w-p.c)*qjoin-p.K
    joins=farmer_if_join>=0
    return {'info':info,'rule':rule,'m':m,'e':e,'units':qjoin if joins else 0.0,
            'farmer_joins':joins,'farmer_net':farmer_if_join if joins else 0.0,
            'farmer_net_if_join':farmer_if_join,
            'intermediary_profit':intermediary_profit(m,e,p,verified) if joins else 0.0}

def factorial(K=12,rho_R=0.0):
    p=Params(K=float(K),rho_R=float(rho_R))
    return [evaluate(rule,info,p) for info in ('Sparse','Verified') for rule in ('Fixed','Chosen')]


In [ ]:
# Baseline two-by-two: K=12, rho_R=0 until behavioral evidence exists.
rho_R=0.0
for row in factorial(K=12,rho_R=rho_R):
    print({k:(round(v,2) if isinstance(v,float) else v) for k,v in row.items()})

In [ ]:
# Export reproducible outputs to CSV files in the Colab session.
from pathlib import Path
import csv

out_dir = Path('outputs')
out_dir.mkdir(exist_ok=True)

def to_export_row(row, K=None):
    exported = {
        'information': row['info'],
        'rule': row['rule'],
        'margin': row['m'],
        'effort': row['e'],
        'completed_units': row['units'],
        'farmer_joins': row['farmer_joins'],
        'farmer_net_if_join': row['farmer_net_if_join'],
        'farmer_net': row['farmer_net'],
        'intermediary_profit': row['intermediary_profit'],
        'assumed_reputation_lift': rho_R,
    }
    if K is not None:
        exported['coordination_cost_K'] = K
    return exported

def save_csv(path, rows):
    with path.open('w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)

factorial_rows = [to_export_row(row) for row in factorial(K=12, rho_R=rho_R)]
sensitivity_rows = [
    to_export_row(row, K=K)
    for K in range(46)
    for row in factorial(K=K, rho_R=rho_R)
    if row['info'] == 'Sparse'
]
threshold_rows = []
for rule in ('Fixed', 'Chosen'):
    p0 = Params(K=0, rho_R=0.0)
    m, e = optimize_fixed(p0) if rule == 'Fixed' else optimize_chosen(p0)
    threshold_rows.append({
        'rule': rule, 'margin': m, 'effort': e,
        'units_before_participation': quantity(m, e, p0),
        'max_K_for_participation': (p0.w-p0.c)*quantity(m, e, p0),
    })

save_csv(out_dir / 'factorial_2x2.csv', factorial_rows)
save_csv(out_dir / 'coordination_cost_sensitivity.csv', sensitivity_rows)
save_csv(out_dir / 'participation_thresholds.csv', threshold_rows)
(out_dir / 'run_log.txt').write_text(
    'Standard-library model run; outputs are synthetic, not field evidence.\n'
    f'Assumed reputation demand lift rho_R: {rho_R:.4f}\n'
    'Factorial cells vary margin rule and seller information; sparse cells use rho_R=0.\n'
    'Price, product-quality evidence, and delivery are held fixed in the proposed experiment.\n'
    'Baseline farmer participation thresholds (K): Fixed 43.2; Chosen 38.4.\n'
    'Change rho_R only after estimating it from the planned behavioral study.\n',
    encoding='utf-8'
)
print('Saved files:', ', '.join(str(x) for x in sorted(out_dir.iterdir())))
print('Rows: factorial =', len(factorial_rows), '; coordination sensitivity =', len(sensitivity_rows), '; thresholds =', len(threshold_rows))


## Optional calibration after the experiment

Estimate the change in purchase probability between verified and sparse seller-history cells while keeping the displayed price and product/quality evidence identical. Map that treatment estimate to the proportional demand shifter `rho_R`, set it in `Params`, and rerun the factorial and K sensitivity. Until those data exist, any nonzero value (for example 0.10) is only a scenario assumption, not an empirical estimate.

Run all notebook cells to write the reproducible CSV outputs to the Colab `outputs/` folder. These files are generated from the model; they are not behavioral-study data.